# BoltzProt-1 binder design — disconnect-proof Colab runner

**The key fact that makes this safe:** the Boltz API runs the compute on *their* servers. This notebook is only a client that submits jobs and collects results. So a dropped Colab session does **not** kill your run — the jobs keep going.

The only thing you can actually lose is the **job IDs**. So this notebook writes every job ID to Google Drive the instant it is created, before doing anything else. If the session dies, you reopen the notebook, hit *Run all*, and it picks up exactly where it left off.

**Everything here is idempotent.** Re-running never resubmits a job you already have and never re-downloads a result you already saved.

### First-time setup
1. Add your Boltz key in the left sidebar: **🔑 Secrets → + New secret → name it `BOLTZ_API_KEY`**, paste the value, toggle *Notebook access* on.
2. Put your target `.cif` somewhere in Drive and set `TARGET_CIF` in the config cell.
3. Runtime → Change runtime type → **CPU is fine** (all the GPU work happens server-side; paying for a GPU here wastes compute units).
4. If you want it to keep running with the laptop shut: Colab Pro+ **background execution** (the toggle appears next to the RAM/Disk gauge). Pro alone will not do this.

> The SDK calls below follow the Boltz API docs as of Aug 2026. If an argument name has drifted, `submit_one()` is the only place you need to touch.

In [ ]:
#@title 1. Install + mount Drive
!pip -q install boltz-api

from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
#@title 2. Config — edit these
from pathlib import Path

# Where everything is written. Must be inside Drive so it survives the VM.
RUN_DIR    = Path('/content/drive/MyDrive/boltzprot/5g51_run01')

# Target structure (CIF). Upload it to Drive first.
TARGET_CIF = Path('/content/drive/MyDrive/boltzprot/targets/5G51_target.cif')

# One job per entry. Give each a stable, unique name — the name IS the identity
# used for resume, so never reuse a name for different settings.
JOBS = [
    {'name': 'nanobody_01', 'binder_specification': {'curated_specification': 'nanobody'}},
    {'name': 'nanobody_02', 'binder_specification': {'curated_specification': 'nanobody'}},
]

POLL_SECONDS   = 30      # how often to check job status
MAX_HOURS      = 23      # stop politely before Colab's 24h ceiling

RESULTS_DIR = RUN_DIR / 'results'
MANIFEST    = RUN_DIR / 'manifest.json'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('Run dir:', RUN_DIR)
assert TARGET_CIF.exists(), f'Target CIF not found: {TARGET_CIF}'

In [ ]:
#@title 3. Auth (reads Colab Secrets — never paste the key into a cell)
import os
from google.colab import userdata

os.environ['BOLTZ_API_KEY'] = userdata.get('BOLTZ_API_KEY')

from boltz_api import Boltz
client = Boltz()   # picks up BOLTZ_API_KEY from the environment
print('Client ready.')

In [ ]:
#@title 4. Manifest — the thing that makes a disconnect survivable
import json, os, tempfile
from datetime import datetime, timezone

def _now():
    return datetime.now(timezone.utc).isoformat(timespec='seconds')

def load_manifest():
    if MANIFEST.exists():
        with open(MANIFEST) as f:
            return json.load(f)
    return {}

def save_manifest(m):
    """Atomic write. A plain open()+write() that dies mid-flush leaves a
    truncated manifest, which loses every job ID in it — the one failure this
    notebook exists to prevent. Write to a temp file, then rename."""
    MANIFEST.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp = tempfile.mkstemp(dir=str(MANIFEST.parent), suffix='.tmp')
    try:
        with os.fdopen(fd, 'w') as f:
            json.dump(m, f, indent=1)
            f.flush()
            os.fsync(f.fileno())
        os.replace(tmp, MANIFEST)
    except BaseException:
        if os.path.exists(tmp):
            os.unlink(tmp)
        raise

manifest = load_manifest()
print(f'{len(manifest)} job(s) already tracked')
for k, v in manifest.items():
    print(f"  {k:20s} {v.get('status','?'):12s} {v.get('job_id','')}")

In [ ]:
#@title 5. Submit (skips anything already submitted)
def submit_one(spec):
    """The only cell that talks to the design endpoint. If the SDK's argument
    names change, this is the single place to fix."""
    return client.compute.v1.protein.design.start(
        target_structure=str(TARGET_CIF),
        binder_specification=spec['binder_specification'],
    )

for spec in JOBS:
    name = spec['name']
    if name in manifest and manifest[name].get('job_id'):
        print(f'skip {name} — already submitted ({manifest[name]["job_id"]})')
        continue
    try:
        job = submit_one(spec)
        # Persist IMMEDIATELY, before anything else can fail. An un-recorded
        # job is a job you pay for and can never collect.
        manifest[name] = {'job_id': job.id, 'status': 'submitted',
                          'submitted_at': _now(), 'spec': spec}
        save_manifest(manifest)
        print(f'submitted {name} -> {job.id}')
    except Exception as e:
        manifest[name] = {'job_id': None, 'status': 'submit_failed',
                          'error': repr(e)[:500], 'submitted_at': _now(), 'spec': spec}
        save_manifest(manifest)
        print(f'FAILED {name}: {e}')

print(f'\n{sum(1 for v in manifest.values() if v.get("job_id"))} job(s) live')

In [ ]:
#@title 6. Poll + download — safe to interrupt and re-run at any time
import time

TERMINAL_OK   = {'completed', 'succeeded', 'success'}
TERMINAL_BAD  = {'failed', 'error', 'cancelled', 'canceled'}

def already_done(name):
    return manifest.get(name, {}).get('status') == 'downloaded'

def fetch_results(name, job_id):
    res = client.compute.v1.protein.design.list_results(job_id)
    designs = []
    for d in getattr(res, 'results', res):
        designs.append({k: v for k, v in vars(d).items()
                        if isinstance(v, (str, int, float, bool, type(None)))}
                       if hasattr(d, '__dict__') else {'raw': str(d)})
    out = RESULTS_DIR / f'{name}.json'
    tmp = out.with_suffix('.json.tmp')
    with open(tmp, 'w') as f:
        json.dump({'name': name, 'job_id': job_id,
                   'downloaded_at': _now(), 'designs': designs}, f, indent=1)
    os.replace(tmp, out)          # atomic, same reason as the manifest
    return len(designs)

deadline = time.time() + MAX_HOURS * 3600

while time.time() < deadline:
    pending = [n for n, v in manifest.items()
               if v.get('job_id') and not already_done(n)]
    if not pending:
        print('\nAll jobs collected.')
        break

    for name in pending:
        jid = manifest[name]['job_id']
        try:
            st = client.compute.v1.protein.design.retrieve(jid)
            status = str(getattr(st, 'status', 'unknown')).lower()
        except Exception as e:
            # transient API/network blip — leave it pending and retry next pass
            print(f'  {name}: poll error ({e}); will retry')
            continue

        if manifest[name].get('status') != status:
            manifest[name]['status'] = status
            manifest[name]['checked_at'] = _now()
            save_manifest(manifest)

        if status in TERMINAL_OK:
            try:
                n = fetch_results(name, jid)
                manifest[name]['status'] = 'downloaded'
                manifest[name]['n_designs'] = n
                save_manifest(manifest)
                print(f'  {name}: downloaded {n} designs')
            except Exception as e:
                print(f'  {name}: download failed ({e}); will retry')
        elif status in TERMINAL_BAD:
            manifest[name]['status'] = 'downloaded'   # terminal; stop polling
            manifest[name]['failed'] = True
            save_manifest(manifest)
            print(f'  {name}: job {status}')

    still = [n for n, v in manifest.items()
             if v.get('job_id') and not already_done(n)]
    if still:
        print(f'[{_now()}] waiting on {len(still)}: {", ".join(still[:6])}')
        time.sleep(POLL_SECONDS)
else:
    print('Hit MAX_HOURS. Jobs are still running server-side — '
          'just re-run this cell later to collect them.')

In [ ]:
#@title 7. Flatten everything collected so far into one CSV
import pandas as pd, glob

rows = []
for p in sorted(glob.glob(str(RESULTS_DIR / '*.json'))):
    with open(p) as f:
        blob = json.load(f)
    for i, d in enumerate(blob.get('designs', [])):
        rows.append({'job_name': blob['name'], 'job_id': blob['job_id'],
                     'design_idx': i, **d})

if rows:
    df = pd.DataFrame(rows)
    out = RUN_DIR / 'all_designs.csv'
    df.to_csv(out, index=False)
    print(f'{len(df)} designs -> {out}')
    display(df.head(20))
else:
    print('Nothing collected yet.')

## If the session dies

Reopen this notebook and **Run all**. Cells 1-4 rebuild state from `manifest.json` on Drive, cell 5 skips everything already submitted, and cell 6 resumes polling. You will not resubmit or double-pay.

## Cost note

Because the heavy compute is server-side, keep this runtime on **CPU**. A GPU runtime burns Colab compute units (~12-15/hour on an A100) to do nothing but `time.sleep`. On CPU the same wait costs a fraction of that.

## Checking on it from anywhere

`manifest.json` is plain JSON in Drive — open it on your phone to see live status without touching Colab.